## OpenAPI Tools with Foundry Agent Service  

### Installing Required Libraries

In [4]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity jsonref

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Setting Up the Environment Variables

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
)
import jsonref

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

### Setting Up the Foundry Project Client

In [2]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Initializing the OpenAPI Tool

In [5]:
with open("./weather_openapi.json", "r") as f:
        openapi_weather = jsonref.loads(f.read())

# Initialize agent OpenApi tool using the read in OpenAPI spec
weather_tool = {
        "type": "openapi",
        "openapi":{
            "name": "weather",
            "spec": openapi_weather,
            "auth": {
                "type": "anonymous"
            },
        }
}

### Setting Up Our Agent with OpenAPI Tool Spec

In [4]:
agent_name = "weather-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful Weather Agent that provides weather information using the provided OpenAPI Tool",
        tools=[weather_tool]
    ),
)

# printing the agent id
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: weather-agent:1, name: weather-agent, version: 1)


### Creating a Conversation Object for the Agent Chat System

In [6]:
# creating an openai client first
openai_client = project_client.get_openai_client()

# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

Created conversation with id: conv_4f3b53722dffa51500mkp7ahKF3cLfYEs9m3x4xYLa2Re7PV6o


In [7]:
user_input = "What's the weather like in New York City today?"

### Chat with The Agent

In [8]:
response = openai_client.responses.create(
            conversation=conversation.id,
            extra_body = {
                "agent": {
                    "name": agent_name,
                    "type": "agent_reference"
                }
            },
            input = user_input
        )
print(f"Weather Agent: {response.output_text}")

Weather Agent: Today in New York City, it's overcast with a temperature of about 2°C (35°F). It feels like -2°C (29°F) with a humidity of 78%. There is a 100% cloud cover, and the wind is coming from the northeast at 12 km/h (8 mph). Visibility is good at 16 km (9 miles), and there is no precipitation predicted. The maximum temperature today is expected to reach 5°C (42°F).
